# 00 — Run the pipeline

This is the operational notebook for routine execution. Use **Run All** after reviewing the single **User settings** cell. It calls the production scripts in order rather than duplicating their implementation.

With the default settings it performs the complete currently available automatic workflow:

1. load or create the manifest;
2. automatically separate GFAP/DAPI channels;
3. detect nucleus instances and create nucleus context inputs;
4. generate automatic GFAP pseudo labels;
5. separate bootstrap individual cells and compartments;
6. print reports and display all QC outputs.

Training, evaluation, and trained-model prediction are included as optional guarded stages. They remain disabled until human-validated annotations and independent data splits exist. Use notebooks 01–04 only when learning, tuning, or troubleshooting a particular stage.

## Environment setup

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p.resolve() for p in candidates if (p / 'pyproject.toml').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Open this notebook from the repository root or notebooks directory.')
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Project root:', PROJECT_ROOT)
print('Python:', sys.executable)

## User settings

For the current test image, keep these defaults and choose **Run All**. `REBUILD_MANIFEST=True` recreates the manifest from `data/raw/` and can discard annotation metadata, so enable it only for a new dataset or intentional reset.

In [ ]:
# Input and automatic-output paths
RAW_DIRECTORY = Path('data/raw')
MANIFEST_PATH = Path('data/metadata/manifest.csv')
INTERIM_DIRECTORY = Path('data/interim')
PSEUDO_DIRECTORY = Path('outputs/pseudo_labels')
PSEUDO_MANIFEST = PSEUDO_DIRECTORY / 'manifest.csv'
INSTANCE_BOOTSTRAP_DIRECTORY = Path('outputs/astrocyte_instances')
INSTANCE_BOOTSTRAP_MANIFEST = INSTANCE_BOOTSTRAP_DIRECTORY / 'manifest.csv'

# Automatic stages
REBUILD_MANIFEST = False
RUN_PREPARATION = True
RUN_GFAP_BOOTSTRAP = True
RUN_INSTANCE_BOOTSTRAP = True
OVERWRITE_AUTOMATIC_OUTPUTS = True
DISPLAY_IMAGE_ID = None  # None displays the first manifest image.

# Optional trained-model stages
TRAIN_CONFIG_PATH = Path('configs/train_instances.yaml')
TRAINING_MANIFEST_OVERRIDE = None  # Example: Path('data/metadata/manifest_instances.csv')
MODEL_OUTPUT_DIRECTORY = Path('outputs/checkpoints/astrocyte_instances')
RUN_TRAINING = False
RUN_EVALUATION = False
RUN_PREDICTION = False
EVALUATION_SPLIT = 'val'
PREDICTION_SPLIT = 'test'
PREDICTION_DIRECTORY = Path('outputs/instance_predictions')
PREDICTION_MANIFEST = PREDICTION_DIRECTORY / 'manifest.csv'

## Production command helper

In [ ]:
def run_script(script, *arguments):
    command = [sys.executable, str(script), *map(str, arguments)]
    print()
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)
    return command

## 1. Load or build the manifest

A missing manifest is built automatically. When a manifest already exists, new BMP/TIFF files are reported but not silently appended because doing so could invalidate experimental metadata and split assignments.

In [ ]:
from astroseg.io import load_manifest

raw_images = sorted(p for p in RAW_DIRECTORY.rglob('*') if p.is_file() and p.suffix.lower() in {'.bmp', '.tif', '.tiff'})
if not raw_images:
    raise FileNotFoundError('Place at least one .bmp, .tif, or .tiff file under data/raw/.')
if REBUILD_MANIFEST or not MANIFEST_PATH.is_file():
    run_script(
        'scripts/build_manifest.py',
        '--raw-dir', RAW_DIRECTORY,
        '--output', MANIFEST_PATH,
    )

manifest = load_manifest(MANIFEST_PATH)
manifest_source_paths = {Path(value).resolve() for value in manifest['path']}
unlisted_images = [path for path in raw_images if path.resolve() not in manifest_source_paths]
print(f'Raw images: {len(raw_images)}; manifest rows: {len(manifest)}')
if unlisted_images:
    print('WARNING: BMP/TIFF files not present in the active manifest:')
    for path in unlisted_images:
        print(' -', path)
    print('Create/merge manifest rows before expecting these files to run.')
manifest

## 2. Run automatic preparation and GFAP segmentation

Preparation updates channel and nucleus-label fields in the source manifest. GFAP proposals are written to a separate pseudo manifest and never replace human annotations.

In [ ]:
if RUN_PREPARATION:
    run_script(
        'scripts/prepare_dataset.py',
        '--manifest', MANIFEST_PATH,
        '--output-dir', INTERIM_DIRECTORY,
    )
else:
    print('Preparation skipped.')

manifest = load_manifest(MANIFEST_PATH)
unannotated_count = int((manifest['annotation_status'] == 'none').sum())
if RUN_GFAP_BOOTSTRAP and unannotated_count:
    bootstrap_arguments = [
        '--manifest', MANIFEST_PATH,
        '--output-dir', PSEUDO_DIRECTORY,
        '--output-manifest', PSEUDO_MANIFEST,
    ]
    if OVERWRITE_AUTOMATIC_OUTPUTS:
        bootstrap_arguments.append('--overwrite')
    run_script('scripts/generate_bootstrap_pseudo_labels.py', *bootstrap_arguments)
elif RUN_GFAP_BOOTSTRAP:
    print('GFAP bootstrap skipped: the source manifest has no rows in annotation_status=none.')
else:
    print('GFAP bootstrap disabled.')

if RUN_INSTANCE_BOOTSTRAP and PSEUDO_MANIFEST.is_file():
    instance_arguments = [
        '--manifest', PSEUDO_MANIFEST,
        '--output-dir', INSTANCE_BOOTSTRAP_DIRECTORY,
        '--output-manifest', INSTANCE_BOOTSTRAP_MANIFEST,
    ]
    if OVERWRITE_AUTOMATIC_OUTPUTS:
        instance_arguments.append('--overwrite')
    run_script('scripts/generate_astrocyte_instances.py', *instance_arguments)
elif RUN_INSTANCE_BOOTSTRAP:
    print('Instance bootstrap skipped: no pseudo GFAP manifest exists.')
else:
    print('Instance bootstrap disabled.')

## 3. Automatic-run summary and QC

In [ ]:
import pandas as pd

preparation_report_path = INTERIM_DIRECTORY / 'qc/preparation_report.csv'
bootstrap_report_path = PSEUDO_DIRECTORY / 'bootstrap_report.csv'
instance_report_path = INSTANCE_BOOTSTRAP_DIRECTORY / 'instance_report.csv'
preparation_report = pd.read_csv(preparation_report_path) if preparation_report_path.is_file() else pd.DataFrame()
bootstrap_report = pd.read_csv(bootstrap_report_path) if bootstrap_report_path.is_file() else pd.DataFrame()
instance_report = pd.read_csv(instance_report_path) if instance_report_path.is_file() else pd.DataFrame()
print('Preparation report:')
print(preparation_report.to_string(index=False) if not preparation_report.empty else 'not created')
print()
print('GFAP bootstrap report:')
print(bootstrap_report.to_string(index=False) if not bootstrap_report.empty else 'not created')
print()
print('Complete-cell bootstrap report (pseudo, watershed ownership):')
print(instance_report.to_string(index=False) if not instance_report.empty else 'not created')

In [ ]:
import matplotlib.pyplot as plt

display_id = DISPLAY_IMAGE_ID or str(manifest.iloc[0]['image_id'])
nucleus_qc_path = INTERIM_DIRECTORY / 'qc' / f'{display_id}_montage.png'
gfap_qc_path = PSEUDO_DIRECTORY / 'overlays' / f'{display_id}.png'
instance_qc_path = INSTANCE_BOOTSTRAP_DIRECTORY / 'overlays' / f'{display_id}_instances.png'
compartment_qc_path = INSTANCE_BOOTSTRAP_DIRECTORY / 'overlays' / f'{display_id}_compartments.png'
available_qc = [
    ('Nucleus and channel QC', nucleus_qc_path),
    ('Automatic GFAP QC', gfap_qc_path),
    ('Bootstrap individual cells', instance_qc_path),
    ('Bootstrap compartments', compartment_qc_path),
]
available_qc = [(title, path) for title, path in available_qc if path.is_file()]
if available_qc:
    figure, axes = plt.subplots(1, len(available_qc), figsize=(9 * len(available_qc), 8), squeeze=False)
    for axis, (title, path) in zip(axes.flat, available_qc):
        axis.imshow(plt.imread(path))
        axis.set_title(f'{title}: {display_id}')
        axis.axis('off')
    figure.tight_layout()
    plt.show()
else:
    print('No QC images were created for', display_id)

## 4. Optional trained-model stages

This cell prepares the complete-cell instance configuration and audits the active training manifest. Real training requires `seed`, `corrected`, or `reviewed` rows with a non-empty `instance_annotation_path`, plus independent train/validation groups.

In [ ]:
import copy
import yaml

from astroseg.constants import TRAINABLE_ANNOTATION_STATUSES

with TRAIN_CONFIG_PATH.open('r', encoding='utf-8') as handle:
    runtime_configuration = copy.deepcopy(yaml.safe_load(handle))
if TRAINING_MANIFEST_OVERRIDE is not None:
    runtime_configuration['data']['manifest_path'] = str(Path(TRAINING_MANIFEST_OVERRIDE))
runtime_configuration['output']['directory'] = str(MODEL_OUTPUT_DIRECTORY)
runtime_config_path = Path('outputs/notebook_run/pipeline_config.yaml')
runtime_config_path.parent.mkdir(parents=True, exist_ok=True)
with runtime_config_path.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(runtime_configuration, handle, sort_keys=False)

training_manifest_path = Path(runtime_configuration['data']['manifest_path'])
training_manifest = load_manifest(training_manifest_path)
trainable = training_manifest['annotation_status'].isin(TRAINABLE_ANNOTATION_STATUSES)
trainable &= training_manifest['instance_annotation_path'].str.strip() != ''
cross_validation = runtime_configuration.get('cross_validation', {})
if cross_validation.get('enabled', False):
    group_column = str(cross_validation.get('group_column', 'image_id'))
    required_groups = int(cross_validation.get('n_splits', 5))
    if group_column in training_manifest.columns:
        group_count = training_manifest.loc[trainable, group_column].str.strip().replace('', pd.NA).nunique()
    else:
        group_count = 0
    training_ready = group_count >= required_groups
    readiness = f'{group_count} valid groups; {required_groups} required'
else:
    train_count = int((trainable & (training_manifest['split'] == 'train')).sum())
    val_count = int((trainable & (training_manifest['split'] == 'val')).sum())
    training_ready = train_count > 0 and val_count > 0
    readiness = f'{train_count} train image(s), {val_count} validation image(s)'
print('Runtime config:', runtime_config_path)
print('Training ready:', training_ready, '-', readiness)

In [ ]:
checkpoint_path = MODEL_OUTPUT_DIRECTORY / 'best.pt'
if RUN_TRAINING:
    if not training_ready:
        raise RuntimeError(f'Training prerequisites are not satisfied: {readiness}')
    run_script('scripts/train_instances.py', '--config', runtime_config_path)
else:
    print('Training disabled.')

if RUN_PREDICTION:
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f'Checkpoint not found: {checkpoint_path}')
    if not (training_manifest['split'] == PREDICTION_SPLIT).any():
        raise RuntimeError(f'No images assigned to split {PREDICTION_SPLIT!r}')
    run_script(
        'scripts/predict_astrocyte_instances.py',
        '--config', runtime_config_path,
        '--checkpoint', checkpoint_path,
        '--split', PREDICTION_SPLIT,
        '--output-dir', PREDICTION_DIRECTORY,
        '--output-manifest', PREDICTION_MANIFEST,
        '--overwrite',
    )
else:
    print('Trained-model prediction disabled.')

if RUN_EVALUATION:
    if not PREDICTION_MANIFEST.is_file():
        raise FileNotFoundError(f'Prediction manifest not found: {PREDICTION_MANIFEST}')
    run_script(
        'scripts/evaluate_astrocyte_instances.py',
        '--manifest', PREDICTION_MANIFEST,
        '--output-dir', 'outputs/metrics/instances',
    )
else:
    print('Instance evaluation disabled.')

## Finished

For the current test image, a successful Run All ends with nucleus QC, a GFAP proposal, individual-cell IDs, and compartment overlays. The cell ownership is still a `watershed_bootstrap`, not human truth. Once complete-cell corrections and independent folds exist, enable the trained instance-model switches.

In [ ]:
print('Automatic pipeline complete.')
print('Prepared manifest:', MANIFEST_PATH)
print('Preparation QC:', INTERIM_DIRECTORY / 'qc')
print('Pseudo manifest:', PSEUDO_MANIFEST if PSEUDO_MANIFEST.is_file() else 'not created')
print('GFAP pseudo outputs:', PSEUDO_DIRECTORY)
print('Bootstrap individual cells:', INSTANCE_BOOTSTRAP_DIRECTORY)
print('Trained instance predictions:', PREDICTION_DIRECTORY)
print('Model checkpoint:', checkpoint_path if checkpoint_path.is_file() else 'not trained yet')